In [135]:
import numpy as np
import pandas as pd 
import matplotlib.pyplot as plt 
import seaborn as sb 
import pyodbc

In [136]:
df_db_customer = pd.read_excel(
    'CustomerChurn_Clean.xlsx', sheet_name='Customers'
)
df_db_subscription = pd.read_excel(
    'CustomerChurn_Clean.xlsx', sheet_name='Subscription'
)
df_db_support = pd.read_excel('CustomerChurn_Clean.xlsx', sheet_name='Support')

In [137]:
df_db_customer.head()
df_db_subscription.head()
df_db_support.head()

,customerid,complaint_date,escalations,csat_score,comment
0,0003-MKNFE,2024-08-28,N,60,service issue
1,0003-MKNFE,2024-08-28,Y,10,demaned refund
2,0013-EXCHZ,2024-01-20,Y,20,NaN
3,0013-MHZWF,2025-03-18,N,90,guidance to renew
4,0013-SMEOE,2024-11-01,N,30,NaN


In [138]:
#Add a Churn Flag
df_db_subscription['churn_flag'] = np.where(df_db_subscription['cancellation_date'].notna(), 1, 0)

In [139]:
df_db_subscription['churn_flag']

0     0
1     1
2     0
3     0
4     1
5     0
6     1
7     0
8     0
9     0
10    0
11    1
12    0
13    1
14    0
15    0
16    0
17    0
18    1
19    0
20    0
Name: churn_flag, dtype: int32

In [140]:
df = (df_db_subscription.merge(df_db_customer, on='customerid', how='left').merge(df_db_support, on='customerid', how='left'))
df.shape

(23, 21)

In [141]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 23 entries, 0 to 22
Data columns (total 21 columns):
 #   Column                   Non-Null Count  Dtype         
---  ------                   --------------  -----         
 0   customerid               23 non-null     object        
 1   subscription_start_date  23 non-null     datetime64[ns]
 2   subscription_type        23 non-null     object        
 3   renewal_date             23 non-null     datetime64[ns]
 4   plan_type                23 non-null     object        
 5   contract_type            23 non-null     object        
 6   cancellation_date        8 non-null      datetime64[ns]
 7   cancellation_reason      8 non-null      object        
 8   monthly_charges          23 non-null     float64       
 9   cltv                     23 non-null     int64         
 10  churn_score              23 non-null     int64         
 11  churn_flag               23 non-null     int32         
 12  customer_name            23 non-null  

In [142]:
df_db_support['customerid'].size    # 9
df_db_support['customerid'].nunique()   # No of Unique: 7
df_db_support['customerid'].unique()   # 7 
#This is why it was giving 23
# df_db_support['customerid']

array(['0003-MKNFE', '0013-EXCHZ', '0013-MHZWF', '0013-SMEOE',
       '0017-IUDMW', '0019-EFAEP', '0022-TCJCI'], dtype=object)

In [143]:
#Add a new column to track complain count of a customer
df_db_support['complain_count'] = df_db_support.groupby('customerid')['customerid'].transform('count')
df_db_support.head()

,customerid,complaint_date,escalations,csat_score,comment,complain_count
0,0003-MKNFE,2024-08-28,N,60,service issue,2
1,0003-MKNFE,2024-08-28,Y,10,demaned refund,2
2,0013-EXCHZ,2024-01-20,Y,20,NaN,1
3,0013-MHZWF,2025-03-18,N,90,guidance to renew,1
4,0013-SMEOE,2024-11-01,N,30,NaN,1


In [144]:
#Keeping only the latest complain and drop the rest
df_db_support = df_db_support.sort_values('complaint_date').drop_duplicates('customerid', keep='last')

In [145]:
df = (df_db_subscription.merge(df_db_customer, on='customerid', how='left').merge(df_db_support, on='customerid', how='left'))
df.shape

(21, 22)

In [146]:
##          ANALYSIS STARTS BELOW           ##
df.to_csv('exported_churn_data.csv', index=False)

In [147]:
df.columns

Index(['customerid', 'subscription_start_date', 'subscription_type',
       'renewal_date', 'plan_type', 'contract_type', 'cancellation_date',
       'cancellation_reason', 'monthly_charges', 'cltv', 'churn_score',
       'churn_flag', 'customer_name', 'country', 'state', 'gender', 'dob',
       'complaint_date', 'escalations', 'csat_score', 'comment',
       'complain_count'],
      dtype='object')

In [148]:
#1. Churn Rate
df['churn_flag'].mean()
churn_rate =  df['churn_flag'].mean() * 100 #in Percentile
print("Churn Rate = ", round(churn_rate, 2), "%")

Churn Rate =  28.57 %


In [149]:
#2. Retention Rate
retention_rate = 100 - churn_rate
print('Retention Rate : ', round(retention_rate, 2), "%")

Retention Rate :  71.43 %


In [150]:
#3. Churn by Plan Type
df.groupby('plan_type')['churn_flag'].mean()
churn_by_plan = df.groupby('plan_type')['churn_flag'].mean().mul(100).round(2).reset_index(name='churn_rate_per')
churn_by_plan

,plan_type,churn_rate_per
0,Basic,60.00
1,Premium,14.29
2,Standard,22.22


In [151]:
#4.1. Churn by State 
churn_by_state = df.groupby('state')['churn_flag'].mean().mul(100).round(2).reset_index(name='churn_rate_per')
churn_by_state

,state,churn_rate_per
0,Delhi,25.00
1,Karnataka,100.00
2,Kathmandu,0.00
3,Maharashtra,0.00
4,Meghalaya,66.67
5,Nagaland,0.00
6,Rajasthan,0.00
7,Telangana,50.00
8,Uttar Pradesh,0.00


In [152]:
#4.1. Churn by Subscription Type
churn_by_subs = df.groupby('subscription_type')['churn_flag'].mean().mul(100).round(2).reset_index(name='churn_rate_per')
churn_by_subs

,subscription_type,churn_rate_per
0,Organic,0.00
1,Paid,16.67
2,Refferal,83.33


In [153]:
# sum(revenue) & count of users
df.head()
rev = df.groupby('state')['monthly_charges'].sum().div(100).round(2)
rev
count_by_state = df.groupby('state')['monthly_charges'].count()
count_by_state

state
Delhi            4
Karnataka        2
Kathmandu        2
Maharashtra      3
Meghalaya        3
Nagaland         1
Rajasthan        2
Telangana        2
Uttar Pradesh    2
Name: monthly_charges, dtype: int64

In [154]:
#5. ARPU - Average Revenue Per User
arpu = df['monthly_charges'].mean()
print('ARPU: ', round(arpu, 2))

ARPU:  18.85


In [155]:
#6. Avg Customer Tenure
df.head() 
toDay = pd.Timestamp.today()
df['tenure_days'] = np.where(
    df['cancellation_date'].notna(), 
    (df['cancellation_date'] - df['subscription_start_date']).dt.days,
    (toDay - df['subscription_start_date']).dt.days
)

df['tenure_days']
avg_tenure = df['tenure_days'].mean()
print('Average tenure days: ', round(avg_tenure))

Average tenure days:  1576


In [156]:
#Revenue At Risk - revenue losst from churn users
revn_at_risk = df.loc[df['churn_flag']==1, 'monthly_charges'].sum()
print('Revenue at Risk (Rs \'K\') =', revn_at_risk)

Revenue at Risk (Rs 'K') = 73.94


In [157]:
#Esclation Rate (thsi a an import KPI)
easclation_rate = (df['escalations'] == 'Y').mean() * 100
print('Esclation rate = ', round(easclation_rate, 2), "%")

Esclation rate =  19.05 %


In [158]:
# Avg complain rate per user
avg_comp_byUser = df['complain_count'].sum() / df['customerid'].nunique()
print('Average Complaint per User =', round(avg_comp_byUser, 2))

Average Complaint per User = 0.43


In [159]:
# Correlate Escelation vs Churn
df[['escalations', 'churn_flag']].dropna()

# encoding string to int
df['escalations'] = np.where(df['escalations'] == 'Y', 1, 0) #
corr_df = df[['escalations', 'churn_flag']]

In [160]:
corr = corr_df['escalations'].corr(df['churn_flag'])
print('Correlation between escelation and churn is = ', round(corr, 2))

Correlation between escelation and churn is =  0.77


In [164]:
#Adding a new column by Churn risk
df['churn_score'].head()
conditions = [
    (df['churn_score'] < 50),
    (df['churn_score'] >= 50) & (df['churn_score'] < 70),
    (df['churn_score'] >= 70)
]

choices = ['low', 'mid', 'high']
df['churn_risk'] = np.select(conditions, choices, default='unknown')
df[['churn_score', 'churn_risk']]

,churn_score,churn_risk
0,12,low
1,91,high
2,34,low
3,8,low
4,88,high
5,22,low
6,79,high
7,5,low
8,34,low
9,41,low


In [163]:
df.head()

,customerid,subscription_start_date,subscription_type,renewal_date,plan_type,contract_type,cancellation_date,cancellation_reason,monthly_charges,cltv,...,state,gender,dob,complaint_date,escalations,csat_score,comment,complain_count,tenure_days,churn_risk
0,0002-ORFBO,2021-03-15,Refferal,2025-03-15,Standard,Annual,NaT,NaN,13.99,627,...,Maharashtra,Male,1982-04-12,NaT,0,NaN,NaN,NaN,2061.0,low
1,0003-MKNFE,2020-08-01,Paid,2024-08-01,Premium,Annual,2024-09-10,Switched to competitor,12.99,1150,...,Karnataka,Male,1995-11-23,2024-08-28,1,10.0,demaned refund,2.0,1501.0,high
2,0004-TLHLJ,2022-11-20,Organic,2025-11-20,Basic,Monthly,NaT,NaN,6.99,210,...,Delhi,Female,1978-02-15,NaT,0,NaN,NaN,NaN,1446.0,low
3,0011-IGKFF,2019-05-10,Paid,2025-05-10,Premium,Annual,NaT,NaN,22.99,1725,...,Nagaland,Male,2001-08-30,NaT,0,NaN,NaN,NaN,2736.0,low
4,0013-EXCHZ,2023-01-05,Refferal,2024-01-05,Standard,Monthly,2024-02-28,Too expensive,13.99,195,...,Delhi,Female,1990-05-05,2024-01-20,1,20.0,NaN,1.0,419.0,high
